# RetailMind AI ? Day 9: Demand Forecasting with an LSTM

This notebook has all the code for preparing sales, training and testing an LSTM, making next-day forecasts, and plotting results. It does not import the Day 9 Python module.

## Before starting

Use Rossmann Store Sales (`Date`, `Store`, `Sales`) or Store Item Demand (`date`, `store`, `item`, `sales`). Put the `train.csv` file in the corresponding subfolder under `data/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)
LOOKBACK = 28
BATCH_SIZE = 256

# Find a Rossmann or Store Item Demand training CSV.
DATA_CSV = None
files = [DATA_CSV, DATA_DIR / "rossmann-store-sales" / "train.csv", DATA_DIR / "store-item-demand-forecasting" / "train.csv", DATA_DIR / "train.csv"]
csv_path = next((Path(p) for p in files if p is not None and Path(p).is_file()), None)
if csv_path is None: raise FileNotFoundError("Put Rossmann train.csv in data/rossmann-store-sales/ or Store Item Demand train.csv in data/store-item-demand-forecasting/.")
data = pd.read_csv(csv_path)
columns = {str(name).strip().lower(): name for name in data.columns}
DATE = columns.get("date")
SALES = columns.get("sales")
STORE = columns.get("store", columns.get("store_id"))
ITEM = columns.get("item", columns.get("item_id"))
if DATE is None or SALES is None or STORE is None: raise ValueError("CSV needs Date, Store, Sales columns; item is optional.")
GROUP_COLUMNS = [STORE] + ([ITEM] if ITEM is not None else [])
data[DATE] = pd.to_datetime(data[DATE], errors="coerce")
data[SALES] = pd.to_numeric(data[SALES], errors="coerce")
data = data.dropna(subset=[DATE, SALES, *GROUP_COLUMNS])
print("Rows:", len(data), "Series columns:", GROUP_COLUMNS)

## 1. Prepare daily sales windows

The code sums sales from the same day and inserts zero-sales days when dates are missing. It uses the oldest 70% of days to train, the next 10% to validate, and the newest 20% to test. Each example uses 28 days to predict the next day.

In [ ]:
# Sum duplicate daily sales and fill days without sales with zero.
series = []
for key, group in data.groupby(GROUP_COLUMNS):
    if not isinstance(key, tuple): key = (key,)
    daily = group.set_index(DATE)[SALES].resample("D").sum()
    dates = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
    sales = daily.reindex(dates, fill_value=0).to_numpy(dtype="float32")
    train_end = int(len(sales) * 0.7)
    valid_end = int(len(sales) * 0.8)
    if train_end <= LOOKBACK or valid_end <= train_end or len(sales) <= valid_end: continue
    mean = float(sales[:train_end].mean())
    scale = float(sales[:train_end].std()) or 1.0
    series.append({"key": dict(zip(GROUP_COLUMNS, key)), "dates": dates, "sales": sales, "scaled": (sales - mean) / scale, "mean": mean, "scale": scale, "train_end": train_end, "valid_end": valid_end})
if not series: raise ValueError("No series are long enough for a 28-day look-back and chronological test split.")

# Make examples: 28 past days are the input, and the next day is the answer.
def make_examples(part):
    windows, targets, actual, dates_out, series_numbers = [], [], [], [], []
    for number, one in enumerate(series):
        if part == "train": start, end = LOOKBACK, one["train_end"]
        elif part == "validation": start, end = one["train_end"], one["valid_end"]
        else: start, end = one["valid_end"], len(one["sales"])
        if end <= start: continue
        all_windows = np.lib.stride_tricks.sliding_window_view(one["scaled"], LOOKBACK + 1)
        first = start - LOOKBACK
        last = end - LOOKBACK
        chosen = all_windows[first:last]
        windows.append(chosen[:, :-1])
        targets.append(chosen[:, -1])
        actual.extend(one["sales"][start:end])
        dates_out.extend(one["dates"][start:end])
        series_numbers.extend([number] * (end - start))
    X = np.concatenate(windows).astype("float32")[..., np.newaxis]
    y = np.concatenate(targets).astype("float32")
    return X, y, np.asarray(actual, dtype="float32"), dates_out, np.asarray(series_numbers)

X_train, y_train, _, _, _ = make_examples("train")
X_valid, y_valid, _, _, _ = make_examples("validation")
X_test, y_test, actual_sales, test_dates, test_series = make_examples("test")
print("Series:", len(series), "Train windows:", len(X_train), "Validation:", len(X_valid), "Test:", len(X_test))

## 2. Train and measure the LSTM

MAE is the average forecast error. RMSE gives larger errors more weight. Smaller scores mean closer predictions.

In [ ]:
# The LSTM learns to predict one day of demand from the previous 28 days.
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(LOOKBACK, 1)),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1),
])
model.compile(optimizer="adam", loss="mse")
model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=20, batch_size=BATCH_SIZE, callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)], verbose=2)

# Restore the original sales units and calculate MAE and RMSE.
predicted_scaled = model.predict(X_test, verbose=0).ravel()
predicted_sales = np.array([max(0.0, predicted_scaled[i] * series[number]["scale"] + series[number]["mean"]) for i, number in enumerate(test_series)])
mae = float(np.mean(np.abs(actual_sales - predicted_sales)))
rmse = float(np.sqrt(np.mean((actual_sales - predicted_sales) ** 2)))
print("Test MAE:", round(mae, 3))
print("Test RMSE:", round(rmse, 3))

# Save the model, scores, and held-out predictions.
model.save(MODEL_DIR / "day9_demand_forecaster_notebook.keras")
pd.DataFrame([{"dataset": csv_path.parent.name, "series_count": len(series), "test_mae": mae, "test_rmse": rmse, "lookback_days": LOOKBACK}]).to_csv(REPORT_DIR / "day9_demand_notebook_metrics.csv", index=False)
predictions = pd.DataFrame([series[number]["key"] for number in test_series])
predictions["date"] = pd.to_datetime(test_dates).strftime("%Y-%m-%d")
predictions["actual_sales"] = actual_sales
predictions["predicted_sales"] = predicted_sales
predictions.to_csv(REPORT_DIR / "day9_demand_notebook_test_predictions.csv", index=False)

# Forecast the day after the last date for every store or store-item series.
next_rows = []
for one in series:
    last_28_days = one["scaled"][-LOOKBACK:][None, :, None]
    next_scaled = float(model.predict(last_28_days, verbose=0)[0][0])
    row = dict(one["key"])
    row["forecast_date"] = (one["dates"][-1] + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    row["forecast_sales"] = max(0.0, next_scaled * one["scale"] + one["mean"])
    next_rows.append(row)
pd.DataFrame(next_rows).to_csv(REPORT_DIR / "day9_demand_notebook_next_day.csv", index=False)

# Plot actual and predicted sales for one series.
example_number = int(test_series[0])
example_rows = predictions.iloc[np.where(test_series == example_number)[0]].sort_values("date").tail(120)
plt.plot(example_rows["date"], example_rows["actual_sales"], label="Actual")
plt.plot(example_rows["date"], example_rows["predicted_sales"], label="Predicted")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig(REPORT_DIR / "day9_demand_notebook_forecast.png")
plt.show()

## Files saved

- Model: `models/day9_demand_forecaster_notebook.keras`
- MAE/RMSE: `reports/day9_demand_notebook_metrics.csv`
- Test predictions: `reports/day9_demand_notebook_test_predictions.csv`
- Next-day forecasts: `reports/day9_demand_notebook_next_day.csv`
- Plot: `reports/day9_demand_notebook_forecast.png`